In [1]:
# ============================================================
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ
# ============================================================

!pip install -q ultralytics sahi

import os
import json
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ============================================================
# 2. ПУТИ
# ============================================================

BASE_PATH   = Path("/kaggle/input/competitions/dl-lab-2-stuff-detection")
TEST_IMAGES = BASE_PATH / "test_images" / "test_images"
SAMPLE_SUB  = BASE_PATH / "sample_sub.csv"
RUNS_DIR    = Path("/kaggle/working/runs")

WEIGHTS     = Path("/kaggle/input/datasets/tkachenko1van/yolo11m-weights/best.pt")

In [3]:
# ============================================================
# 3. ЗАГРУЗКА МОДЕЛИ
# ============================================================

model = YOLO(str(WEIGHTS))
print(f"✅ Модель загружена: {WEIGHTS}")

✅ Модель загружена: /kaggle/input/datasets/tkachenko1van/yolo11m-weights/best.pt


In [4]:
# ============================================================
# 4. SAHI ИНФЕРЕНС
# ============================================================

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(WEIGHTS),
    confidence_threshold=0.25,
    device="cuda:0",
    image_size=640,
)

IMG_W, IMG_H = 1280, 720
STAFF_NAME = "staff"
sample_sub = pd.read_csv(str(SAMPLE_SUB))

rows = []
total_boxes = 0

for idx, row in sample_sub.iterrows():
    img_name = str(row["image_name"])
    img_path = str(TEST_IMAGES / img_name)

    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=480,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        perform_standard_pred=True,
        postprocess_type="NMS",
        postprocess_match_metric="IOU",
        postprocess_match_threshold=0.5,
        verbose=0,
    )

    boxes = []
    for pred in result.object_prediction_list:
        if pred.category.name != STAFF_NAME:
            continue

        conf = pred.score.value
        bbox = pred.bbox

        xc = (bbox.minx + bbox.maxx) / 2 / IMG_W
        yc = (bbox.miny + bbox.maxy) / 2 / IMG_H
        w  = (bbox.maxx - bbox.minx) / IMG_W
        h  = (bbox.maxy - bbox.miny) / IMG_H

        boxes.append([round(xc,6), round(yc,6),
                      round(w,6), round(h,6), round(conf,4)])

    total_boxes += len(boxes)
    rows.append({
        "id": idx,
        "image_name": img_name,
        "boxes": json.dumps(boxes, separators=(",",":")),
    })

    if (idx+1) % 200 == 0:
        print(f"  [{idx+1}/{len(sample_sub)}]")

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)
imgs_with_det = sum(1 for r in rows if r["boxes"] != "[]")
print(f"Submission: {len(sub)} imgs, {imgs_with_det} w/ det, {total_boxes} boxes")

  [200/4454]
  [400/4454]
  [600/4454]
  [800/4454]
  [1000/4454]
  [1200/4454]
  [1400/4454]
  [1600/4454]
  [1800/4454]
  [2000/4454]
  [2200/4454]
  [2400/4454]
  [2600/4454]
  [2800/4454]
  [3000/4454]
  [3200/4454]
  [3400/4454]
  [3600/4454]
  [3800/4454]
  [4000/4454]
  [4200/4454]
  [4400/4454]
Submission: 4454 imgs, 1836 w/ det, 3472 boxes
